In [1]:
import numpy as np
from keras.preprocessing.image import ImageDataGenerator
from keras.applications.resnet50 import ResNet50
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
# Load VGG-16 model without top (fully connected layers)
resnet = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

94765736/94765736 [==============================] - 1s 0us/step


In [ ]:
# Define function to extract features using VGG-16
def extract_features(generator, model):
    features = model.predict(generator, verbose=1)
    return features

In [ ]:
# Directory paths for train and test data
train_dir = "Processed_Dataset_2/train"
test_dir = "Processed_Dataset_2/test"

# Define image size and batch size
image_size = (224, 224)
batch_size = 32

In [ ]:
# Create data generators for train and test data
train_datagen = ImageDataGenerator(rescale=1./255)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode=None,  # No labels are returned
    shuffle=False  # Important: Keep the order of the images
)

In [ ]:
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=image_size,
    batch_size=batch_size,
    class_mode=None,
    shuffle=False
)

In [ ]:
# Extract features for train and test data
train_features = extract_features(train_generator, resnet)
test_features = extract_features(test_generator, resnet)

In [ ]:
# Flatten features
train_features_flatten = np.reshape(train_features, (train_features.shape[0], -1))
test_features_flatten = np.reshape(test_features, (test_features.shape[0], -1))

In [ ]:
# Train Random Forest classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(train_features_flatten, train_generator.classes)

In [ ]:
# Predict using Random Forest classifier
predictions = rf_classifier.predict(test_features_flatten)

In [ ]:
# Calculate accuracy
accuracy = accuracy_score(test_generator.classes, predictions)
print("Accuracy:", accuracy)

In [ ]:
# Calculate confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_generator.classes, predictions)

# Plot confusion matrix
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=test_generator.class_indices.keys(), yticklabels=test_generator.class_indices.keys())
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from sklearn.metrics import classification_report
class_names = ["Normal", "Pneumonia"]
report = classification_report(test_generator.classes, predictions, target_names=class_names)
print(report)